# Gentrification Early Warning System

Load the dataset, import the required libraries, and verify the columns needed by the project.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)
from xgboost import XGBClassifier
from IPython.display import display

# DATASET LOADING: use the expected name when available; otherwise find an uploaded CSV.
DATASET_NAME = "balanced_training_dataset.csv"
possible_paths = [
    f"/content/{DATASET_NAME}",
    DATASET_NAME,
    "/mnt/data/balanced_training_dataset(6).csv"
]
DATASET_PATH = next((p for p in possible_paths if os.path.exists(p)), None)

if DATASET_PATH is None:
    csv_candidates = glob.glob("/content/*.csv") + glob.glob("*.csv")
    csv_candidates = [p for p in csv_candidates if "balanced_training_dataset" in os.path.basename(p).lower()]
    if csv_candidates:
        DATASET_PATH = csv_candidates[0]

if DATASET_PATH is None:
    try:
        from google.colab import files
        print("Upload your balanced_training_dataset CSV file.")
        uploaded = files.upload()
        csv_candidates = [name for name in uploaded if name.lower().endswith('.csv')]
        if not csv_candidates:
            raise FileNotFoundError("No CSV file was uploaded. Please upload balanced_training_dataset.csv.")
        # Prefer the project's expected CSV name, including copies renamed with a suffix.
        matching = [name for name in csv_candidates if "balanced_training_dataset" in name.lower()]
        DATASET_PATH = matching[0] if matching else csv_candidates[0]
    except ImportError:
        raise FileNotFoundError("CSV not found. Place balanced_training_dataset.csv beside this notebook and rerun this cell.")

df = pd.read_csv(DATASET_PATH)
required_columns = ["tract_id", "gentrification"]
missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}")

print("Dataset loaded successfully:", DATASET_PATH)
print("Dataset shape:", df.shape)
print("\nClass distribution before duplicate removal:")
print(df["gentrification"].value_counts(dropna=False))
print("\nColumns:")
print(df.columns.tolist())


## 1. Define the baseline features

The baseline model uses nine socioeconomic and housing indicators from 2012. The target is the binary `gentrification` label.

In [ ]:
features = [
    "total_population_2012",
    "median_household_income_2012",
    "median_gross_rent_2012",
    "median_home_value_2012",
    "pct_higher_education_2012",
    "poverty_rate_2012",
    "vacancy_rate_2012",
    "unemployment_rate_2012",
    "pct_renter_occupied_2012"
]

target = "gentrification"

X = df[features].copy()
y = df[target].copy()

print("Number of features:", len(features))
print("\nFeatures:")
for feature in features:
    print("-", feature)

print("\nTarget:", target)
print("\nMissing values:")
print(X.isnull().sum())

## 2. Inspect duplicate tract records

Check how many records repeat the same Census tract before creating the train/test split.

In [ ]:
duplicate_count = df["tract_id"].duplicated().sum()
unique_tracts = df["tract_id"].nunique()

print("Total rows:", len(df))
print("Unique tracts:", unique_tracts)
print("Duplicate rows:", duplicate_count)
print("\nClass distribution among duplicate rows:")
print(df.loc[df["tract_id"].duplicated(keep="first"), "gentrification"].value_counts())

if df["tract_id"].isna().any():
    print("WARNING: Rows with missing tract_id:", int(df["tract_id"].isna().sum()))


## 3. Clean the dataset and check labels

Keep one record per tract so duplicate tracts cannot appear in both training and testing data. Verify that labels are binary.

In [ ]:
# Keep one row per tract before splitting to prevent duplicated tracts crossing train/test sets.
df_unique = df.drop_duplicates(subset="tract_id", keep="first").copy()
df_unique = df_unique.dropna(subset=["tract_id", target]).copy() if "target" in globals() else df_unique.dropna(subset=["tract_id", "gentrification"]).copy()
df_unique["gentrification"] = pd.to_numeric(df_unique["gentrification"], errors="raise").astype(int)

if not set(df_unique["gentrification"].unique()).issubset({0, 1}):
    raise ValueError("The gentrification target must contain only binary labels 0 and 1.")

print("Rows after removing duplicate tracts:", len(df_unique))
print("Unique tracts:", df_unique["tract_id"].nunique())
print("\nClass distribution:")
print(df_unique["gentrification"].value_counts().sort_index())


## 4. Split the baseline dataset

Create a stratified 80/20 train/test split. The test set is kept separate from training-time balancing.

In [ ]:
X_unique = df_unique[features].copy()
y_unique = df_unique[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_unique,
    y_unique,
    test_size=0.20,
    random_state=42,
    stratify=y_unique
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

## 5. Balance the training set only

Oversample the minority class in the training set only. Do not oversample the test set.

In [ ]:
train_data = pd.concat([X_train, y_train], axis=1)

majority = train_data[train_data[target] == 0]
minority = train_data[train_data[target] == 1]

minority_oversampled = minority.sample(
    n=len(majority),
    replace=True,
    random_state=42
)

train_balanced = pd.concat(
    [majority, minority_oversampled]
).sample(frac=1, random_state=42)

X_train_balanced = train_balanced[features]
y_train_balanced = train_balanced[target]

print("Balanced training samples:", len(X_train_balanced))
print("\nBalanced training class distribution:")
print(y_train_balanced.value_counts())

print("\nTest samples remain:", len(X_test))
print("\nTest class distribution:")
print(y_test.value_counts())

## 6. Impute missing values

Fit the imputer on training data and apply it to both training and test data.

In [ ]:
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train_balanced)
X_test_imputed = imputer.transform(X_test)

print("Training shape:", X_train_imputed.shape)
print("Testing shape:", X_test_imputed.shape)

print("\nRemaining missing values in training:",
      np.isnan(X_train_imputed).sum())

print("Remaining missing values in testing:",
      np.isnan(X_test_imputed).sum())

## 7. Scale features for Logistic Regression

Standardize features for the scale-sensitive Logistic Regression model. Tree-based models use imputed, unscaled data.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

## 8. Train baseline Logistic Regression

Fit and generate predictions from the nine-feature baseline model.

In [ ]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_train_balanced
)

y_pred_logistic = logistic_model.predict(X_test_scaled)
y_prob_logistic = logistic_model.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression trained successfully.")

### Baseline results: Logistic Regression

Review accuracy, precision, recall, F1-score, ROC-AUC, and the classification report.

In [ ]:
print("Logistic Regression Results")
print("-" * 40)

accuracy = accuracy_score(y_test, y_pred_logistic)
precision = precision_score(y_test, y_pred_logistic, zero_division=0)
recall = recall_score(y_test, y_pred_logistic, zero_division=0)
f1 = f1_score(y_test, y_pred_logistic, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob_logistic)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logistic, zero_division=0))

## 9. Train baseline Decision Tree

Fit the baseline Decision Tree.

In [ ]:
decision_tree_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

decision_tree_model.fit(
    X_train_imputed,
    y_train_balanced
)

y_pred_dt = decision_tree_model.predict(X_test_imputed)
y_prob_dt = decision_tree_model.predict_proba(X_test_imputed)[:, 1]

print("Decision Tree trained successfully.")

### Baseline results: Decision Tree

Review the baseline Decision Tree metrics.

In [ ]:
print("Decision Tree Results")
print("-" * 40)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt, zero_division=0)
recall_dt = recall_score(y_test, y_pred_dt, zero_division=0)
f1_dt = f1_score(y_test, y_pred_dt, zero_division=0)
roc_auc_dt = roc_auc_score(y_test, y_prob_dt)

print(f"Accuracy : {accuracy_dt:.4f}")
print(f"Precision: {precision_dt:.4f}")
print(f"Recall   : {recall_dt:.4f}")
print(f"F1-Score : {f1_dt:.4f}")
print(f"ROC-AUC  : {roc_auc_dt:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, zero_division=0))

## 10. Train baseline Random Forest

Fit the baseline Random Forest.

In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_train_imputed,
    y_train_balanced
)

y_pred_rf = random_forest_model.predict(X_test_imputed)
y_prob_rf = random_forest_model.predict_proba(X_test_imputed)[:, 1]

print("Random Forest trained successfully.")

### Baseline results: Random Forest

Review the baseline Random Forest metrics.

In [ ]:
print("Random Forest Results")
print("-" * 40)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf, zero_division=0)
recall_rf = recall_score(y_test, y_pred_rf, zero_division=0)
f1_rf = f1_score(y_test, y_pred_rf, zero_division=0)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1-Score : {f1_rf:.4f}")
print(f"ROC-AUC  : {roc_auc_rf:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

## 11. Train baseline XGBoost

Fit the baseline XGBoost model.

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train_imputed,
    y_train_balanced
)

y_pred_xgb = xgb_model.predict(X_test_imputed)
y_prob_xgb = xgb_model.predict_proba(X_test_imputed)[:, 1]

print("XGBoost trained successfully.")

### Baseline results: XGBoost

Review the baseline XGBoost metrics.

In [ ]:
print("XGBoost Results")
print("-" * 40)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb, zero_division=0)
recall_xgb = recall_score(y_test, y_pred_xgb, zero_division=0)
f1_xgb = f1_score(y_test, y_pred_xgb, zero_division=0)
roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)

print(f"Accuracy : {accuracy_xgb:.4f}")
print(f"Precision: {precision_xgb:.4f}")
print(f"Recall   : {recall_xgb:.4f}")
print(f"F1-Score : {f1_xgb:.4f}")
print(f"ROC-AUC  : {roc_auc_xgb:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, zero_division=0))

## 12. Baseline confusion matrices

Compare true and predicted labels for all four nine-feature models. Rows represent actual labels; columns represent predicted labels.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

models = [
    ("Logistic Regression", y_pred_logistic, axes[0, 0]),
    ("Decision Tree", y_pred_dt, axes[0, 1]),
    ("Random Forest", y_pred_rf, axes[1, 0]),
    ("XGBoost", y_pred_xgb, axes[1, 1])
]

for name, predictions, ax in models:
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        predictions,
        ax=ax,
        cmap="Blues"
    )
    ax.set_title(name)

plt.tight_layout()
plt.show()

## 13. Baseline ROC curves

ROC curves compare the trade-off between true-positive rate and false-positive rate across thresholds.

In [ ]:
from sklearn.metrics import roc_curve

# Calculate ROC curves
fpr_log, tpr_log, _ = roc_curve(y_test, y_prob_logistic)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)

plt.figure(figsize=(9, 7))

plt.plot(
    fpr_log, tpr_log,
    label=f"Logistic Regression (AUC = {roc_auc:.3f})"
)

plt.plot(
    fpr_dt, tpr_dt,
    label=f"Decision Tree (AUC = {roc_auc_dt:.3f})"
)

plt.plot(
    fpr_rf, tpr_rf,
    label=f"Random Forest (AUC = {roc_auc_rf:.3f})"
)

plt.plot(
    fpr_xgb, tpr_xgb,
    label=f"XGBoost (AUC = {roc_auc_xgb:.3f})"
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(True)
plt.show()

## 14. Random Forest feature importance

This model-specific importance estimates which baseline features the Random Forest used most.

In [ ]:
feature_importance_rf = pd.DataFrame({
    "Feature": features,
    "Importance": random_forest_model.feature_importances_
})

feature_importance_rf = feature_importance_rf.sort_values(
    by="Importance",
    ascending=False
)

print("Random Forest Feature Importance")
print("-" * 50)
print(feature_importance_rf.to_string(index=False))

### Visualize baseline feature importance

Inspect the feature-importance chart.

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    feature_importance_rf["Feature"],
    feature_importance_rf["Importance"]
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

## 15. Permutation importance for the baseline

Permutation importance measures how test performance changes when a feature is shuffled.

In [ ]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    random_forest_model,
    X_test_imputed,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc"
)

permutation_importance_rf = pd.DataFrame({
    "Feature": features,
    "Importance_Mean": perm_result.importances_mean,
    "Importance_Std": perm_result.importances_std
})

permutation_importance_rf = permutation_importance_rf.sort_values(
    by="Importance_Mean",
    ascending=False
)

print("Random Forest Permutation Importance")
print("-" * 60)
print(permutation_importance_rf.to_string(index=False))

## 16. Engineer two additional features

Add home-value-to-income and annual-rent-to-income ratios. These are the two extra features that extend the model from nine to eleven inputs.

In [ ]:
df_unique["home_value_to_income_2012"] = (
    df_unique["median_home_value_2012"] /
    df_unique["median_household_income_2012"]
)

df_unique["annual_rent_to_income_2012"] = (
    (df_unique["median_gross_rent_2012"] * 12) /
    df_unique["median_household_income_2012"]
)

print("New features created:")
print("- home_value_to_income_2012")
print("- annual_rent_to_income_2012")

print("\nSample values:")
print(
    df_unique[
        [
            "home_value_to_income_2012",
            "annual_rent_to_income_2012"
        ]
    ].head()
)

## 17. Prepare the 11-feature experiment

Use the same unique tracts, target, stratified split settings, and training-only balancing strategy for a fair comparison.

In [ ]:
extra_features = [
    "home_value_to_income_2012",
    "annual_rent_to_income_2012"
]

features_11 = features + extra_features

X_unique_11 = df_unique[features_11].copy()
y_unique_11 = df_unique[target].copy()

X_train_11, X_test_11, y_train_11, y_test_11 = train_test_split(
    X_unique_11,
    y_unique_11,
    test_size=0.20,
    random_state=42,
    stratify=y_unique_11
)

print("Number of features:", len(features_11))
print("\nFeatures:")
for feature in features_11:
    print("-", feature)

print("\nTraining samples:", len(X_train_11))
print("Testing samples:", len(X_test_11))

print("\nTraining class distribution:")
print(y_train_11.value_counts())

print("\nTesting class distribution:")
print(y_test_11.value_counts())

## 18. Balance the enhanced training set

Oversample only the enhanced model training set; keep the enhanced test set untouched.

In [ ]:
train_data_11 = pd.concat([X_train_11, y_train_11], axis=1)

majority_11 = train_data_11[train_data_11[target] == 0]
minority_11 = train_data_11[train_data_11[target] == 1]

minority_oversampled_11 = minority_11.sample(
    n=len(majority_11),
    replace=True,
    random_state=42
)

train_balanced_11 = pd.concat(
    [majority_11, minority_oversampled_11]
).sample(frac=1, random_state=42)

X_train_balanced_11 = train_balanced_11[features_11]
y_train_balanced_11 = train_balanced_11[target]

print("Balanced training samples:", len(X_train_balanced_11))

print("\nBalanced training class distribution:")
print(y_train_balanced_11.value_counts())

print("\nTest samples remain:", len(X_test_11))

print("\nTest class distribution:")
print(y_test_11.value_counts())

## 19. Impute missing values for 11 features

Fit a separate imputer for the enhanced experiment.

In [ ]:
imputer_11 = SimpleImputer(strategy="median")

X_train_imputed_11 = imputer_11.fit_transform(X_train_balanced_11)
X_test_imputed_11 = imputer_11.transform(X_test_11)

print("Training shape:", X_train_imputed_11.shape)
print("Testing shape:", X_test_imputed_11.shape)

print("\nRemaining missing values in training:",
      np.isnan(X_train_imputed_11).sum())

print("Remaining missing values in testing:",
      np.isnan(X_test_imputed_11).sum())

## 20. Scale enhanced features for Logistic Regression

Fit a separate scaler on enhanced training data.

In [ ]:
scaler_11 = StandardScaler()

X_train_scaled_11 = scaler_11.fit_transform(X_train_imputed_11)
X_test_scaled_11 = scaler_11.transform(X_test_imputed_11)

print("Scaled training shape:", X_train_scaled_11.shape)
print("Scaled testing shape:", X_test_scaled_11.shape)

## 21. Train and evaluate all four 11-feature models

Train Logistic Regression, Decision Tree, Random Forest, and XGBoost using the enhanced feature set. Review the metrics generated from this run.

In [ ]:
# ============================================
# TRAIN + EVALUATE ALL 4 MODELS (11 FEATURES)
# ============================================

# 1. Logistic Regression
logistic_model_11 = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model_11.fit(
    X_train_scaled_11,
    y_train_balanced_11
)

y_pred_logistic_11 = logistic_model_11.predict(X_test_scaled_11)
y_prob_logistic_11 = logistic_model_11.predict_proba(X_test_scaled_11)[:, 1]


# 2. Decision Tree
decision_tree_model_11 = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

decision_tree_model_11.fit(
    X_train_imputed_11,
    y_train_balanced_11
)

y_pred_dt_11 = decision_tree_model_11.predict(X_test_imputed_11)
y_prob_dt_11 = decision_tree_model_11.predict_proba(X_test_imputed_11)[:, 1]


# 3. Random Forest
random_forest_model_11 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_forest_model_11.fit(
    X_train_imputed_11,
    y_train_balanced_11
)

y_pred_rf_11 = random_forest_model_11.predict(X_test_imputed_11)
y_prob_rf_11 = random_forest_model_11.predict_proba(X_test_imputed_11)[:, 1]


# 4. XGBoost
xgb_model_11 = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_model_11.fit(
    X_train_imputed_11,
    y_train_balanced_11
)

y_pred_xgb_11 = xgb_model_11.predict(X_test_imputed_11)
y_prob_xgb_11 = xgb_model_11.predict_proba(X_test_imputed_11)[:, 1]


# ============================================
# CALCULATE RESULTS
# ============================================

results_11 = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],

    "Accuracy": [
        accuracy_score(y_test_11, y_pred_logistic_11),
        accuracy_score(y_test_11, y_pred_dt_11),
        accuracy_score(y_test_11, y_pred_rf_11),
        accuracy_score(y_test_11, y_pred_xgb_11)
    ],

    "Precision": [
        precision_score(y_test_11, y_pred_logistic_11, zero_division=0),
        precision_score(y_test_11, y_pred_dt_11, zero_division=0),
        precision_score(y_test_11, y_pred_rf_11, zero_division=0),
        precision_score(y_test_11, y_pred_xgb_11, zero_division=0)
    ],

    "Recall": [
        recall_score(y_test_11, y_pred_logistic_11, zero_division=0),
        recall_score(y_test_11, y_pred_dt_11, zero_division=0),
        recall_score(y_test_11, y_pred_rf_11, zero_division=0),
        recall_score(y_test_11, y_pred_xgb_11, zero_division=0)
    ],

    "F1-Score": [
        f1_score(y_test_11, y_pred_logistic_11, zero_division=0),
        f1_score(y_test_11, y_pred_dt_11, zero_division=0),
        f1_score(y_test_11, y_pred_rf_11, zero_division=0),
        f1_score(y_test_11, y_pred_xgb_11, zero_division=0)
    ],

    "ROC-AUC": [
        roc_auc_score(y_test_11, y_prob_logistic_11),
        roc_auc_score(y_test_11, y_prob_dt_11),
        roc_auc_score(y_test_11, y_prob_rf_11),
        roc_auc_score(y_test_11, y_prob_xgb_11)
    ]
})

print("RESULTS WITH 11 FEATURES")
print("=" * 80)
print(results_11.round(4).to_string(index=False))

### Enhanced-model confusion matrices

These are separate from the baseline confusion matrices above. They use predictions from the eleven-feature models, so the plot lets you compare whether the added ratios change classification errors.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

models_11 = [
    ("Logistic Regression — 11 features", y_pred_logistic_11),
    ("Decision Tree — 11 features", y_pred_dt_11),
    ("Random Forest — 11 features", y_pred_rf_11),
    ("XGBoost — 11 features", y_pred_xgb_11),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for (name, predictions), ax in zip(models_11, axes.ravel()):
    ConfusionMatrixDisplay.from_predictions(
        y_test_11, predictions, labels=[0, 1],
        display_labels=["No gentrification", "Gentrification"],
        cmap="Blues", values_format="d", ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.suptitle("Confusion Matrices — Enhanced 11-Feature Models", y=1.02)
plt.tight_layout()
plt.show()


## 22. Enhanced Decision Tree permutation importance

Inspect which of the eleven features most affect the enhanced Decision Tree F1-score on the test set.

In [ ]:
from IPython.display import display
from sklearn.inspection import permutation_importance
import pandas as pd
import matplotlib.pyplot as plt

perm = permutation_importance(
    decision_tree_model_11,
    X_test_imputed_11,
    y_test_11,
    n_repeats=10,
    random_state=42,
    scoring="f1"
)

perm_df = pd.DataFrame({
    "Feature": X_test_11.columns,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values("Importance", ascending=False)

print("Permutation Importance - Enhanced Decision Tree")
display(perm_df)

plt.figure(figsize=(9, 5))
plt.barh(
    perm_df["Feature"],
    perm_df["Importance"]
)
plt.xlabel("Mean decrease in F1-score")
plt.ylabel("Feature")
plt.title("Permutation Feature Importance - Enhanced Decision Tree")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 23. Compare all eight experiments

Compare each model using nine features against the same model using eleven features. Use the displayed metrics rather than manually copied numbers.

In [ ]:
# Compare the 9-feature and 11-feature experiments using metrics from this notebook run.
def metric_row(label, y_true, y_pred, y_prob):
    return {
        "Experiment": label,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob) if len(set(y_true)) == 2 else np.nan
    }

comparison_rows = [
    metric_row("Logistic Regression — 9 features", y_test, y_pred_logistic, y_prob_logistic),
    metric_row("Decision Tree — 9 features", y_test, y_pred_dt, y_prob_dt),
    metric_row("Random Forest — 9 features", y_test, y_pred_rf, y_prob_rf),
    metric_row("XGBoost — 9 features", y_test, y_pred_xgb, y_prob_xgb),
    metric_row("Logistic Regression — 11 features", y_test_11, y_pred_logistic_11, y_prob_logistic_11),
    metric_row("Decision Tree — 11 features", y_test_11, y_pred_dt_11, y_prob_dt_11),
    metric_row("Random Forest — 11 features", y_test_11, y_pred_rf_11, y_prob_rf_11),
    metric_row("XGBoost — 11 features", y_test_11, y_pred_xgb_11, y_prob_xgb_11)
]
final_comparison = pd.DataFrame(comparison_rows)
print("MODEL COMPARISON — CURRENT NOTEBOOK RUN")
display(final_comparison.round(4))
print("\nNote: The test set contains only 10 positive-labelled tracts; interpret positive-class metrics cautiously.")


## 24. Live demonstration: enhanced Decision Tree

Enter values for the nine original features. The notebook calculates the two ratios and passes all eleven features to the trained enhanced Decision Tree. Use realistic values and explain that this is a demonstration, not a validated real-world planning decision.

In [ ]:
# ============================================
# LIVE DEMO - GENTRIFICATION EARLY WARNING
# ============================================

import numpy as np
import pandas as pd

print("=== GENTRIFICATION EARLY WARNING SYSTEM ===")
print("Enter 2012 Census Tract values\n")

# --------------------------------------------
# Feature names used by the final 11-feature model
# --------------------------------------------

features = [
    "total_population_2012",
    "median_household_income_2012",
    "median_gross_rent_2012",
    "median_home_value_2012",
    "pct_higher_education_2012",
    "poverty_rate_2012",
    "vacancy_rate_2012",
    "unemployment_rate_2012",
    "pct_renter_occupied_2012"
]

extra_features = [
    "home_value_to_income_2012",
    "annual_rent_to_income_2012"
]

features_11 = features + extra_features

# --------------------------------------------
# Enter the 9 original features
# --------------------------------------------

total_population = float(input("Total Population: "))
median_household_income = float(input("Median Household Income: "))
median_gross_rent = float(input("Median Gross Rent: "))
median_home_value = float(input("Median Home Value: "))
pct_higher_education = float(input("Higher Education (%): "))
poverty_rate = float(input("Poverty Rate (%): "))
vacancy_rate = float(input("Vacancy Rate (%): "))
unemployment_rate = float(input("Unemployment Rate (%): "))
pct_renter_occupied = float(input("Renter Occupied (%): "))

# --------------------------------------------
# Validate input values
# --------------------------------------------

if not np.all(np.isfinite([
    total_population,
    median_household_income,
    median_gross_rent,
    median_home_value,
    pct_higher_education,
    poverty_rate,
    vacancy_rate,
    unemployment_rate,
    pct_renter_occupied
])):
    raise ValueError("Please enter valid numeric values.")

if total_population <= 0:
    raise ValueError("Total Population must be greater than 0.")

if median_household_income <= 0:
    raise ValueError("Median Household Income must be greater than 0.")

if median_gross_rent < 0:
    raise ValueError("Median Gross Rent cannot be negative.")

if median_home_value < 0:
    raise ValueError("Median Home Value cannot be negative.")

percentage_values = {
    "Higher Education": pct_higher_education,
    "Poverty Rate": poverty_rate,
    "Vacancy Rate": vacancy_rate,
    "Unemployment Rate": unemployment_rate,
    "Renter Occupied": pct_renter_occupied
}

for name, value in percentage_values.items():
    if value < 0 or value > 100:
        raise ValueError(f"{name} must be between 0 and 100.")

# --------------------------------------------
# Create the two engineered features
# --------------------------------------------

home_value_to_income = (
    median_home_value / median_household_income
)

annual_rent_to_income = (
    (median_gross_rent * 12) / median_household_income
)

# --------------------------------------------
# Create input using the same 11 features
# used by the final model
# --------------------------------------------

live_input = pd.DataFrame([[
    total_population,
    median_household_income,
    median_gross_rent,
    median_home_value,
    pct_higher_education,
    poverty_rate,
    vacancy_rate,
    unemployment_rate,
    pct_renter_occupied,
    home_value_to_income,
    annual_rent_to_income
]], columns=features_11)

# --------------------------------------------
# Check that the trained model exists
# --------------------------------------------

if "imputer_11" not in globals():
    raise NameError(
        "imputer_11 is not available. Run the 11-feature training cells first."
    )

if "decision_tree_model_11" not in globals():
    raise NameError(
        "decision_tree_model_11 is not available. Run the 11-feature training cells first."
    )

# --------------------------------------------
# Apply the SAME imputer used during training
# --------------------------------------------

live_input_imputed = imputer_11.transform(live_input)

# --------------------------------------------
# Final prediction using Enhanced Decision Tree
# --------------------------------------------

prediction = decision_tree_model_11.predict(live_input_imputed)[0]

probability = decision_tree_model_11.predict_proba(
    live_input_imputed
)[0, 1]

# --------------------------------------------
# Display result
# --------------------------------------------

print("\n=== PREDICTION RESULT ===")

if prediction == 1:
    print("⚠️ GENTRIFICATION RISK DETECTED")
else:
    print("✅ NO GENTRIFICATION RISK DETECTED")

print(f"Model class probability: {probability:.2%}")

print("\n=== ENGINEERED FEATURES ===")
print(f"Home Value / Income Ratio: {home_value_to_income:.2f}")
print(f"Annual Rent / Income Ratio: {annual_rent_to_income:.2f}")

## 25. Show a true-positive example

Show an example whose known label is positive and which the enhanced Decision Tree also predicts as positive. This is an illustrative example, not independent validation.

In [ ]:
# Find an actual positive row that the trained model also predicts as 1

positive_rows = df_unique[df_unique["gentrification"] == 1].copy()

positive_rows["home_value_to_income_2012"] = (
    positive_rows["median_home_value_2012"] /
    positive_rows["median_household_income_2012"]
)

positive_rows["annual_rent_to_income_2012"] = (
    positive_rows["median_gross_rent_2012"] * 12 /
    positive_rows["median_household_income_2012"]
)

X_positive = positive_rows[features_11]

X_positive_imputed = imputer_11.transform(X_positive)

positive_rows["model_prediction"] = (
    decision_tree_model_11.predict(X_positive_imputed)
)

true_positive = positive_rows[
    positive_rows["model_prediction"] == 1
]

print("Number of actual positive rows predicted as 1:",
      len(true_positive))

if len(true_positive) > 0:

    sample = true_positive.iloc[0]

    print("\n=== TRUE POSITIVE SAMPLE ===")

    for feature in features:
        print(f"{feature}: {sample[feature]}")

    print("\nEngineered features:")
    print(
        f"Home Value / Income Ratio: "
        f"{sample['home_value_to_income_2012']:.2f}"
    )

    print(
        f"Annual Rent / Income Ratio: "
        f"{sample['annual_rent_to_income_2012']:.2f}"
    )

    print(f"\nActual label: {int(sample['gentrification'])}")
    print(f"Model prediction: {int(sample['model_prediction'])}")

else:
    print("No true-positive sample found.")